# Ch12 RAG 통합과 시각화

Ch12의 실제 문서와 통합 로그를 읽어 검색, 병합, 재정렬, 생성을 하나의 그래프로 연결할 수 있을까요?

## 개념을 코드에 연결하기

**호출 표기와 실행 시점.** 통합 그래프의 초기 상태에는 질문과 PDF 위치가 들어가고 각 노드는 자신이 만든 필드만 반환합니다. load_pages가 documents를 만들면 검색 노드가 그 값을 읽고, fuse와 rerank가 candidates를 좁힌 뒤 generate가 선택 문서로 answer를 만듭니다. trace는 데이터 자체가 아니라 실행 순서를 확인하려고 누적하는 관찰 필드입니다.

**다른 상황에 적용.** 사내 규정 RAG에서는 PDF 대신 위키 문서를 읽고, sparse_search 대신 제목 키워드 검색을 사용할 수 있습니다. 그래도 로드→복수 검색→병합→재정렬→생성이라는 단계 경계와 각 노드의 입출력 이름을 유지하면 한 단계씩 교체하고 평가할 수 있습니다.

**비교와 경계.** 그래프가 완성됐다는 사실은 답변 품질을 보장하지 않습니다. Mermaid는 선언된 연결을 보여 주고 trace는 이번 실행 경로를 보여 줄 뿐입니다. 검색 정확도와 답변 근거성은 Ch15의 평가 데이터와 지표로 따로 확인해야 합니다.


## 실행 환경 준비

Python 3.12 새 커널에서 첫 셀부터 실행합니다. 설치 셀은 이 노트북에 필요한 버전을 고정합니다.

## 패키지 설치와 환경 준비

필요한 패키지를 고정 버전으로 설치합니다. 출력에서 LangGraph 버전 또는 API 키의 존재 여부만 확인합니다.

**이 셀의 코드 읽기.** `for` 블록은 입력 항목을 하나씩 처리하며, 반복이 끝난 뒤 모은 값이 다음 셀의 입력이 됩니다. `print()`는 확인용 문자열을 화면에 표시할 뿐 함수의 반환값을 바꾸지 않습니다. `assert`는 조건이 참이면 출력 없이 통과하고 거짓이면 실행을 중단하므로, 바로 앞 결과를 검증하는 역할을 합니다.


In [1]:
# 노트북을 독립 실행할 수 있도록 패키지와 환경을 준비합니다.
%pip install -q langgraph==1.2.11 langchain==1.4.0 langchain-openai==1.6.1 python-dotenv==1.2.3 numpy==2.5.3 scikit-learn==1.9.0 pypdf==6.18.0 kiwipiepy==0.23.2 rank-bm25==0.2.2 sentence-transformers==6.0.1
import os
from pathlib import Path
from dotenv import load_dotenv

for folder in [Path.cwd(), *Path.cwd().parents]:
    if (folder / '.env').is_file():
        load_dotenv(folder / '.env')
        break
assert os.getenv('OPENAI_API_KEY'), 'OPENAI_API_KEY가 필요합니다.'
print('API 키 존재 여부 확인 완료')

Note: you may need to restart the kernel to use updated packages.
API 키 존재 여부 확인 완료


## Ch12 질문과 문서 목록 읽기

Ch12 질문과 문서 목록 읽기 단계를 실행합니다. 출력된 상태 값과 코드에서 반환한 키를 서로 대조합니다.

**이 셀의 코드 읽기.** `json.loads(text)`는 JSON 문자열을 파이썬 딕셔너리나 목록으로 변환해 반환합니다. `read_text()`는 파일 내용을 문자열로 반환합니다. `print()`는 확인용 문자열을 화면에 표시할 뿐 함수의 반환값을 바꾸지 않습니다.


In [2]:
# Ch12 질문과 문서 목록 읽기 단계의 연산을 실행하고 반환된 상태 값을 확인합니다.
import json
from importlib.metadata import version
BASE = Path('labs/part3/ch12-retrieval-tuning')
run = json.loads((BASE / 'runs/advanced-continuity-integrated-result.json').read_text())
dataset = json.loads((BASE / 'data/advanced-documents.json').read_text())
question = run['question']
parents = dataset['parents']
print('langgraph', version('langgraph'))
print('질문:', question, 'PDF:', len(parents))

langgraph 1.2.11
질문: 도서는 한 번에 몇 권까지, 며칠 동안 빌릴 수 있나요? PDF: 6


## 상태와 PDF 로딩 노드 정의

다음 코드에서 상태와 PDF 로딩 노드 정의에 필요한 형식과 함수를 선언합니다. 이 셀은 정의를 준비하며, 값은 이어지는 실행 셀에서 확인합니다.

**이 셀의 코드 읽기.** 이 셀을 실행하면 `RAGState` 클래스 정의가 준비됩니다. 클래스 본문은 정의 시 처리되지만 인스턴스 값은 나중에 객체를 만들 때 생깁니다. `load_pages(state: RAGState)`는 함수를 정의하는 표기입니다. 이 셀에서는 함수 객체만 준비되고 들여쓴 본문은 뒤에서 함수를 호출할 때 실행됩니다. `return` 뒤의 값은 화면 출력이 아니라 호출한 코드로 전달되는 반환값입니다.


In [3]:
# 상태와 PDF 로딩 노드 정의에 사용할 상태 형식과 함수를 선언합니다.
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END
from pypdf import PdfReader

class RAGState(TypedDict):
    question: str
    parents: list[dict]
    chunks: list[dict]
    dense_scores: list[float]
    sparse_scores: list[float]
    candidates: list[int]
    selected: list[dict]
    answer: str
    usage: dict
    trace: list[str]

def load_pages(state: RAGState) -> dict:
    chunks = []
    for parent in state['parents']:
        pdf = Path('labs/data/rag-corpus') / parent['source']
        for page, obj in enumerate(PdfReader(pdf).pages, 1):
            chunks.append({'chunk_id': f"{parent['doc_id']}-P{page:02d}-C01", 'parent_id': parent['doc_id'], 'page': page, 'text': obj.extract_text() or ''})
    return {'chunks': chunks, 'trace': state['trace'] + ['load_pages']}

## OpenAI 임베딩 검색 노드 정의

다음 코드에서 OpenAI 임베딩 검색 노드 정의에 필요한 형식과 함수를 선언합니다. 이 셀은 정의를 준비하며, 값은 이어지는 실행 셀에서 확인합니다.

**이 셀의 코드 읽기.** `dense_search(state: RAGState)`는 함수를 정의하는 표기입니다. 이 셀에서는 함수 객체만 준비되고 들여쓴 본문은 뒤에서 함수를 호출할 때 실행됩니다. `return` 뒤의 값은 화면 출력이 아니라 호출한 코드로 전달되는 반환값입니다. `OpenAIEmbeddings(...)`는 문자열을 수치 벡터로 변환할 임베딩 클라이언트를 준비합니다. 실제 호출은 `embed_query()`나 `embed_documents()`에서 일어납니다. `embed_documents(texts)`는 여러 문장을 입력받아 문장 수만큼의 벡터를 반환합니다. `embed_query(text)`는 검색 질문 한 개를 벡터 하나로 반환합니다. `assert`는 조건이 참이면 출력 없이 통과하고 거짓이면 실행을 중단하므로, 바로 앞 결과를 검증하는 역할을 합니다.


In [4]:
# OpenAI 임베딩 검색 노드 정의에 사용할 상태 형식과 함수를 선언합니다.
import numpy as np
from langchain_openai import OpenAIEmbeddings
from sklearn.metrics.pairwise import cosine_similarity

def dense_search(state: RAGState) -> dict:
    embedding = OpenAIEmbeddings(model='text-embedding-3-small', dimensions=256)
    texts = [item['text'] for item in state['chunks']]
    doc_vectors = np.array(embedding.embed_documents(texts))
    query_vector = np.array(embedding.embed_query(state['question']))
    scores = cosine_similarity([query_vector], doc_vectors)[0]
    assert doc_vectors.shape == (len(texts), 256)
    return {'dense_scores': scores.tolist(), 'trace': state['trace'] + ['dense_search']}

## Kiwi BM25 검색 노드 정의

다음 코드에서 Kiwi BM25 검색 노드 정의에 필요한 형식과 함수를 선언합니다. 이 셀은 정의를 준비하며, 값은 이어지는 실행 셀에서 확인합니다.

**이 셀의 코드 읽기.** `sparse_search(state: RAGState)`는 함수를 정의하는 표기입니다. 이 셀에서는 함수 객체만 준비되고 들여쓴 본문은 뒤에서 함수를 호출할 때 실행됩니다. `return` 뒤의 값은 화면 출력이 아니라 호출한 코드로 전달되는 반환값입니다. `BM25Okapi(tokenized_documents)`는 토큰 목록으로 희소 검색 점수를 계산할 인덱스를 준비합니다.


In [5]:
# Kiwi BM25 검색 노드 정의에 사용할 상태 형식과 함수를 선언합니다.
from kiwipiepy import Kiwi
from rank_bm25 import BM25Okapi

def sparse_search(state: RAGState) -> dict:
    kiwi = Kiwi()
    tokenized = [[token.form for token in kiwi.tokenize(item['text'])] for item in state['chunks']]
    bm25 = BM25Okapi(tokenized)
    query_tokens = [token.form for token in kiwi.tokenize(state['question'])]
    scores = bm25.get_scores(query_tokens)
    return {'sparse_scores': scores.tolist(), 'trace': state['trace'] + ['sparse_search']}

## RRF 병합 노드 정의

다음 코드에서 RRF 병합 노드 정의에 필요한 형식과 함수를 선언합니다. 이 셀은 정의를 준비하며, 값은 이어지는 실행 셀에서 확인합니다.

**이 셀의 코드 읽기.** `rrf_pair(first, second, rank_constant=60)`, `fuse(state: RAGState)`는 함수를 정의하는 표기입니다. 이 셀에서는 함수 객체만 준비되고 들여쓴 본문은 뒤에서 함수를 호출할 때 실행됩니다. `return` 뒤의 값은 화면 출력이 아니라 호출한 코드로 전달되는 반환값입니다.


In [6]:
# RRF 병합 노드 정의에 사용할 상태 형식과 함수를 선언합니다.
def rrf_pair(first, second, rank_constant=60) -> list[int]:
    scores = {}
    for ranking in (first, second):
        for rank, index in enumerate(ranking, 1):
            scores[index] = scores.get(index, 0.0) + 1 / (rank_constant + rank)
    return sorted(scores, key=scores.get, reverse=True)

def fuse(state: RAGState) -> dict:
    dense_rank = list(np.argsort(-np.array(state['dense_scores']))[:50])
    sparse_rank = list(np.argsort(-np.array(state['sparse_scores']))[:50])
    candidates = rrf_pair(dense_rank, sparse_rank)[:10]
    return {'candidates': candidates, 'trace': state['trace'] + ['fuse']}

## CrossEncoder 재정렬 노드 정의

다음 코드에서 CrossEncoder 재정렬 노드 정의에 필요한 형식과 함수를 선언합니다. 이 셀은 정의를 준비하며, 값은 이어지는 실행 셀에서 확인합니다.

**이 셀의 코드 읽기.** `rerank(state: RAGState)`는 함수를 정의하는 표기입니다. 이 셀에서는 함수 객체만 준비되고 들여쓴 본문은 뒤에서 함수를 호출할 때 실행됩니다. `return` 뒤의 값은 화면 출력이 아니라 호출한 코드로 전달되는 반환값입니다. `CrossEncoder(model_name)`은 질문·문서 쌍을 함께 읽어 재정렬 점수를 계산할 모델을 준비합니다.


In [7]:
# CrossEncoder 재정렬 노드 정의에 사용할 상태 형식과 함수를 선언합니다.
from sentence_transformers import CrossEncoder

def rerank(state: RAGState) -> dict:
    model = CrossEncoder('BAAI/bge-reranker-v2-m3', max_length=256, device='cpu')
    pairs = [(state['question'], state['chunks'][i]['text']) for i in state['candidates']]
    scores = model.predict(pairs, show_progress_bar=False)
    ordered = [state['candidates'][pos] for pos in np.argsort(-scores)]
    selected, seen = [], set()
    for index in ordered:
        item = state['chunks'][index]
        if item['parent_id'] in seen:
            continue
        selected.append(item)
        seen.add(item['parent_id'])
        if len(selected) == 2:
            break
    return {'selected': selected, 'trace': state['trace'] + ['rerank']}

[사용자 폴더]/working/claude/projects/한국경제신문/labs/part4/ch13-langgraph-basics/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 근거 기반 생성 노드 정의

다음 코드에서 근거 기반 생성 노드 정의에 필요한 형식과 함수를 선언합니다. 이 셀은 정의를 준비하며, 값은 이어지는 실행 셀에서 확인합니다.

**이 셀의 코드 읽기.** `generate(state: RAGState)`는 함수를 정의하는 표기입니다. 이 셀에서는 함수 객체만 준비되고 들여쓴 본문은 뒤에서 함수를 호출할 때 실행됩니다. `return` 뒤의 값은 화면 출력이 아니라 호출한 코드로 전달되는 반환값입니다. `invoke(input)`가 나타나는 줄에서 실제 그래프 또는 모델 실행이 시작되고 반환값이 왼쪽 변수에 저장됩니다. `init_chat_model(...)`은 채팅 모델 객체를 만들며, 이 줄에서는 아직 답변을 생성하지 않습니다.


In [8]:
# 근거 기반 생성 노드 정의에 사용할 상태 형식과 함수를 선언합니다.
from langchain.chat_models import init_chat_model

def generate(state: RAGState) -> dict:
    context = '\n\n'.join(f"[{d['parent_id']} p.{d['page']} {d['chunk_id']}]\n{d['text']}" for d in state['selected'])
    prompt = f"""아래 자료만 사용해 질문에 한국어로 답하세요.
답 끝에 [doc_id p.쪽] 형식으로 출처를 표시하세요.

질문: {state['question']}

자료:
{context}"""
    model = init_chat_model('openai:gpt-5.4-mini', temperature=0)
    reply = model.invoke(prompt)
    return {'answer': reply.text.strip(), 'usage': reply.usage_metadata or {}, 'trace': state['trace'] + ['generate']}

## RAG 그래프 컴파일과 실제 실행

노드와 엣지를 연결한 뒤 실행 가능한 그래프로 컴파일합니다. 다음 셀에서 이 그래프를 호출합니다.

**이 셀의 코드 읽기.** `StateGraph(StateType)`은 상태 형식을 입력으로 받아 그래프 설계 객체를 만듭니다. `add_node(name, function)`은 그래프 이름과 실제 실행할 함수를 연결하며, 이 호출만으로 함수 본문이 실행되지는 않습니다. `add_edge(from, to)`는 두 노드의 고정 이동 순서를 등록합니다. `compile()`은 선언한 노드와 엣지로 `invoke()`할 실행 객체를 반환합니다. `invoke(input)`가 나타나는 줄에서 실제 그래프 또는 모델 실행이 시작되고 반환값이 왼쪽 변수에 저장됩니다. `for` 블록은 입력 항목을 하나씩 처리하며, 반복이 끝난 뒤 모은 값이 다음 셀의 입력이 됩니다.


In [9]:
# RAG 그래프 컴파일과 실제 실행 단계의 연산을 실행하고 반환된 상태 값을 확인합니다.
from time import perf_counter
builder = StateGraph(RAGState)
for name, fn in [('load_pages', load_pages), ('dense_search', dense_search), ('sparse_search', sparse_search), ('fuse', fuse), ('rerank', rerank), ('generate', generate)]:
    builder.add_node(name, fn)
builder.add_edge(START, 'load_pages')
builder.add_edge('load_pages', 'dense_search')
builder.add_edge('dense_search', 'sparse_search')
builder.add_edge('sparse_search', 'fuse')
builder.add_edge('fuse', 'rerank')
builder.add_edge('rerank', 'generate')
builder.add_edge('generate', END)
graph = builder.compile()
initial = {'question': question, 'parents': parents, 'chunks': [], 'dense_scores': [], 'sparse_scores': [], 'candidates': [], 'selected': [], 'answer': '', 'usage': {}, 'trace': []}
started = perf_counter()
result = graph.invoke(initial)
duration_ms = round((perf_counter() - started) * 1000, 2)
print('선택:', [(d['parent_id'], d['page'], d['chunk_id']) for d in result['selected']])
print('답변:', result['answer'])
print('실행 흔적:', result['trace'])
assert any(d['parent_id'] == 'E01' and d['page'] == 3 for d in result['selected'])
assert '[E01 p.3]' in result['answer']

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 393/393 [00:00<00:00, 6429.14it/s]

선택: [('E01', 3, 'E01-P03-C01'), ('E02', 3, 'E02-P03-C01')]
답변: 도서는 한 번에 7권까지, 14일 동안 빌릴 수 있습니다. [E01 p.3]
실행 흔적: ['load_pages', 'dense_search', 'sparse_search', 'fuse', 'rerank', 'generate']


## 통합 실행 결과 저장

실제 실행 결과를 다음 챕터가 읽을 JSON 파일로 저장합니다. 출력에서 저장 경로와 실행 시간을 확인합니다.

**이 셀의 코드 읽기.** `write_text(text)`가 실행되는 줄에서 문자열이 실제 파일에 저장됩니다. `for` 블록은 입력 항목을 하나씩 처리하며, 반복이 끝난 뒤 모은 값이 다음 셀의 입력이 됩니다. `print()`는 확인용 문자열을 화면에 표시할 뿐 함수의 반환값을 바꾸지 않습니다.


In [10]:
# 검색·생성 결과와 실행 정보를 재사용할 JSON 파일로 저장합니다.
run_payload = {
    'qid': run['qid'], 'question': question, 'input_ch12_run_id': run['run_id'],
    'retrieved': [{'doc_id': d['parent_id'], 'page': d['page'], 'chunk_id': d['chunk_id']} for d in result['selected']],
    'node_trace': result['trace'], 'response': result['answer'],
    'models': {'embedding': 'text-embedding-3-small', 'dimensions': 256, 'sparse': 'Kiwi+BM25', 'fusion': 'RRF(k=60)', 'reranker': 'BAAI/bge-reranker-v2-m3', 'chat': 'gpt-5.4-mini'},
    'usage': result['usage'], 'duration_ms': duration_ms,
}
run_dir = Path('labs/part4/ch13-langgraph-basics/runs')
run_dir.mkdir(exist_ok=True)
run_path = run_dir / 'graph-integrated-result.json'
run_path.write_text(json.dumps(run_payload, ensure_ascii=False, indent=2))
print('저장:', run_path, '시간:', duration_ms, 'ms')

저장: labs/part4/ch13-langgraph-basics/runs/graph-integrated-result.json 시간: 13186.79 ms


## RRF 상수 하나를 바꿔 비교

RRF 상수 하나를 바꿔 비교 조건으로 값을 하나만 변경합니다. 나란히 출력된 두 결과에서 달라진 항목을 확인합니다.

**이 셀의 코드 읽기.** `for` 블록은 입력 항목을 하나씩 처리하며, 반복이 끝난 뒤 모은 값이 다음 셀의 입력이 됩니다. `print()`는 확인용 문자열을 화면에 표시할 뿐 함수의 반환값을 바꾸지 않습니다.


In [11]:
# 한 조건만 바꾸고 나머지 입력을 유지해 결과 차이를 비교합니다.
dense_rank = list(np.argsort(-np.array(result['dense_scores']))[:50])
sparse_rank = list(np.argsort(-np.array(result['sparse_scores']))[:50])
k60 = rrf_pair(dense_rank, sparse_rank, rank_constant=60)[:3]
k40 = rrf_pair(dense_rank, sparse_rank, rank_constant=40)[:3]
print('k=60:', [result['chunks'][i]['chunk_id'] for i in k60])
print('k=40:', [result['chunks'][i]['chunk_id'] for i in k40])

k=60: ['E01-P03-C01', 'E01-P04-C01', 'E02-P03-C01']
k=40: ['E01-P03-C01', 'E01-P04-C01', 'E02-P03-C01']


## 실제 실행 순서 시각화

컴파일한 그래프에서 Mermaid 문자열을 생성합니다. 출력에서 노드 이름과 연결 방향을 확인합니다.

**이 셀의 코드 읽기.** `draw_mermaid()`는 컴파일된 그래프의 노드와 연결을 Mermaid 문자열로 반환하며 그래프를 다시 실행하지 않습니다. `for` 블록은 입력 항목을 하나씩 처리하며, 반복이 끝난 뒤 모은 값이 다음 셀의 입력이 됩니다. `print()`는 확인용 문자열을 화면에 표시할 뿐 함수의 반환값을 바꾸지 않습니다. `assert`는 조건이 참이면 출력 없이 통과하고 거짓이면 실행을 중단하므로, 바로 앞 결과를 검증하는 역할을 합니다.


In [12]:
# 컴파일된 그래프의 실제 노드와 엣지를 Mermaid 문자열로 확인합니다.
mermaid = graph.get_graph().draw_mermaid()
print(mermaid)
for node in ['load_pages', 'dense_search', 'sparse_search', 'fuse', 'rerank', 'generate']:
    assert node in mermaid

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	load_pages(load_pages)
	dense_search(dense_search)
	sparse_search(sparse_search)
	fuse(fuse)
	rerank(rerank)
	generate(generate)
	__end__([<p>__end__</p>]):::last
	__start__ --> load_pages;
	dense_search --> sparse_search;
	fuse --> rerank;
	load_pages --> dense_search;
	rerank --> generate;
	sparse_search --> fuse;
	generate --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc

